# NB04 — Regional Replication: EUR (GEMAS) and AUS (NGSA)

**Notebook:** microbeatlas_cwm / NB04  
**Primary estimand:** L1 (pH-adjusted) FWL — same as NB02 USA  
**Metals:** As, Cd, Cr, Cu, Ni, Pb, Zn (present in both GEMAS and NGSA joins)  
**KOs tested:** ALL 6,557 CWM KOs (same scope as NB02)  
**Levels:** L0–L5 (L6 skipped: land-cover categorical not available globally)  
**Regions:** EUR (921 MA thinned samples, GEMAS join) · AUS (236 MA samples, NGSA join)  

**Replication criterion:** NB02 USA L1 hit (q<0.05) → EUR or AUS L1 hit (q<0.05) for same KO × metal, concordant direction.


In [1]:
import sys
sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, PALETTE, METAL_COLORS, FIGW, ROW_H
apply_style()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t as t_dist
from scipy.interpolate import BSpline
import patsy
from statsmodels.stats.multitest import multipletests
from pathlib import Path

DATA = Path('/home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_cwm/data')
FIGS = Path('/home/hmacgregor/BERIL-research-observatory/projects/microbeatlas_cwm/figures')
FIGS.mkdir(exist_ok=True)


In [2]:
# Load all base tables
nb00     = pd.read_parquet(DATA / 'nb00_thinned_samples.parquet')
combined = pd.read_parquet(DATA / 'nb02_combined_metals.parquet')
gc       = pd.read_parquet(DATA / 'nb01_genus_counts.parquet')         # genus counts per sample
gp       = pd.read_parquet(DATA / 'nb02_genus_phylum.parquet')         # genus -> phylum mapping
soil     = pd.read_parquet(DATA / 'nb02_soil_props.parquet')
lith     = pd.read_parquet(DATA / 'nb02_lithology.parquet')
mine     = pd.read_parquet(DATA / 'nb02_mine_dist.parquet')
wclim    = pd.read_parquet(DATA / 'nb02_worldclim_seasonal.parquet')
cwm_long = pd.read_parquet(DATA / 'nb01_cwm.parquet')                  # 23M rows, long format

# NB02 USA L1 hits for overlap analysis
usa_fdr = pd.read_parquet(DATA / 'nb02_fwl_results_fdr.parquet')
usa_L1_hits = usa_fdr[(usa_fdr['level']=='L1') & (usa_fdr['q_bh']<0.05)][['metal','ko_id','beta_per_iqr']].copy()
usa_L1_hits['usa_direction'] = np.sign(usa_L1_hits['beta_per_iqr'])
print(f'USA L1 hits: {len(usa_L1_hits)} ({usa_L1_hits["ko_id"].nunique()} KOs, {usa_L1_hits["metal"].nunique()} metals)')

# Build merged base table with lat/lon and pH
base = combined.merge(
    nb00[['sample_id','lat','lon','ph','olm_soil_ph_0cm_H2O','ph_soilgrids',
          'era5_mean_2m_air_temp_k','era5_total_precipitation_mm',
          'dem_elevation_m','ndvi']],
    on='sample_id', how='left'
)
base = base.merge(soil[['sample_id','clay_pct','som_pct','bulk_density']], on='sample_id', how='left')
base = base.merge(lith[['sample_id','lith_dist_deg']], on='sample_id', how='left')
base = base.merge(mine[['sample_id','mine_dist_km']], on='sample_id', how='left')
base = base.merge(wclim[['sample_id','temp_seasonality','precip_seasonality']], on='sample_id', how='left')

# Best-available pH: measured > OLM
ph_raw_b = pd.to_numeric(base['ph'], errors='coerce')
ph_olm_b = pd.to_numeric(base['olm_soil_ph_0cm_H2O'], errors='coerce') / 10.0
base['ph_best'] = ph_raw_b.where(ph_raw_b.notna(),
                  ph_olm_b.where(ph_olm_b.notna(), base['ph_soilgrids']))

print(f'base shape: {base.shape}')
print(f'region counts:\n{base["region"].value_counts()}')


USA L1 hits: 560 (427 KOs, 12 metals)
base shape: (4884, 75)
region counts:
region
EUR    921
USA    545
AUS    236
Name: count, dtype: int64


In [3]:
# Compute Shannon diversity and top-phylum RAs from genus_counts
tot = gc.groupby('sample_id')['genus_count'].sum().rename('total')
gc2 = gc.join(tot, on='sample_id')
gc2['ra'] = gc2['genus_count'] / gc2['total']

# Shannon entropy
gc2['h'] = -gc2['ra'] * np.log(gc2['ra'].clip(lower=1e-12))
shannon = gc2.groupby('sample_id')['h'].sum().rename('shannon')

# Phylum RA: map genus -> phylum
gc2 = gc2.merge(gp, on='genus_lower', how='left')
gc2['phylum_lower'] = gc2['phylum_lower'].fillna('unknown')
phyl_ra = gc2.groupby(['sample_id','phylum_lower'])['ra'].sum().reset_index()

# Top 8 phyla by mean RA across all samples
top_phyla = (phyl_ra.groupby('phylum_lower')['ra'].mean()
             .nlargest(8).index.tolist())
print('Top 8 phyla:', top_phyla)

# Pivot phylum RA wide
phyl_wide = phyl_ra[phyl_ra['phylum_lower'].isin(top_phyla)].pivot_table(
    index='sample_id', columns='phylum_lower', values='ra', fill_value=0.0)
phyl_wide.columns = [f'phyl_{c}' for c in phyl_wide.columns]

# Merge community covariates into base
base = base.merge(shannon.reset_index(), on='sample_id', how='left')
base = base.merge(phyl_wide.reset_index(), on='sample_id', how='left')
phyl_cols = [c for c in base.columns if c.startswith('phyl_')]
base[phyl_cols] = base[phyl_cols].fillna(0.0)

print(f'base with community covariates: {base.shape}')


Top 8 phyla: ['alphaproteobacteria', 'sordariomycetes', 'actinomycetia', 'gammaproteobacteria', 'bryopsida', 'insecta', 'nitrososphaeria', 'wallemiomycetes']
base with community covariates: (4884, 84)


In [4]:
# Filter CWM to EUR and AUS samples; pivot to wide (samples × KOs)
EUR_METALS = ['As','Cd','Cr','Cu','Ni','Pb','Zn']
AUS_METALS = ['As','Cr','Cu','Ni','Pb','Zn']  # Cd n=76 too sparse

eur_base = base[base['region']=='EUR'].set_index('sample_id')
aus_base = base[base['region']=='AUS'].set_index('sample_id')

print('Pivoting CWM for EUR...')
cwm_eur_long = cwm_long[cwm_long['sample_id'].isin(eur_base.index)]
cwm_eur = cwm_eur_long.pivot_table(index='sample_id', columns='ko_id', values='cwm', fill_value=0.0)
cwm_eur = cwm_eur.reindex(eur_base.index).fillna(0.0)
print(f'EUR CWM: {cwm_eur.shape}')

print('Pivoting CWM for AUS...')
cwm_aus_long = cwm_long[cwm_long['sample_id'].isin(aus_base.index)]
cwm_aus = cwm_aus_long.pivot_table(index='sample_id', columns='ko_id', values='cwm', fill_value=0.0)
cwm_aus = cwm_aus.reindex(aus_base.index).fillna(0.0)
print(f'AUS CWM: {cwm_aus.shape}')

ko_ids = cwm_eur.columns.tolist()  # use EUR KO list as reference
print(f'Total KOs: {len(ko_ids)}')


Pivoting CWM for EUR...


EUR CWM: (921, 6516)
Pivoting CWM for AUS...


AUS CWM: (236, 6435)
Total KOs: 6516


In [5]:
# Vectorized FWL — same as NB02
def fwl_all_kos(X, Y, Z):
    """FWL partial regression: X=scalar metal, Y=matrix (n,p_KOs), Z=confounder matrix."""
    n, p = Z.shape
    coef_x, *_ = np.linalg.lstsq(Z, X, rcond=None)
    Mx = X - Z @ coef_x
    coef_y, *_ = np.linalg.lstsq(Z, Y, rcond=None)
    My = Y - Z @ coef_y
    MxMx = Mx @ Mx
    betas = My.T @ Mx / MxMx
    resid = My - np.outer(Mx, betas)
    dof = max(n - p - 1, 1)
    sigma2 = (resid ** 2).sum(axis=0) / dof
    se = np.sqrt(sigma2 / MxMx)
    t_stat = betas / np.where(se > 0, se, np.nan)
    return betas, se, t_stat

def build_Z(sample_df, level):
    """Build confounder matrix for a given causal level."""
    n = len(sample_df)
    intercept = np.ones((n, 1))

    if level == 0:
        return intercept

    ph = sample_df['ph_best'].values.astype(float)
    # Natural spline for pH (3 df)
    ph_vals = np.where(np.isfinite(ph), ph, np.nanmedian(ph))
    try:
        ph_dm = patsy.dmatrix('cr(ph_v, df=3)', {'ph_v': ph_vals},
                               return_type='matrix')[:, 1:]  # drop intercept
    except Exception:
        ph_dm = ph_vals.reshape(-1, 1)
    Z = np.hstack([intercept, ph_dm])

    if level == 1:
        return Z

    # L2: + soil props + lithology
    soil_cols = ['clay_pct','som_pct','bulk_density','lith_dist_deg']
    for c in soil_cols:
        col = pd.to_numeric(sample_df[c], errors='coerce').values
        col = np.where(np.isfinite(col), col, np.nanmedian(col[np.isfinite(col)]) if np.any(np.isfinite(col)) else 0.0)
        Z = np.hstack([Z, col.reshape(-1,1)])

    if level == 2:
        return Z

    # L3: + log mine distance
    mine = pd.to_numeric(sample_df['mine_dist_km'], errors='coerce').values
    mine = np.where(mine > 0, mine, 0.001)
    mine = np.where(np.isfinite(mine), mine, 500.0)
    Z = np.hstack([Z, np.log10(mine).reshape(-1,1)])

    if level == 3:
        return Z

    # L4: + climate (MAT, MAP, temp_seasonality, precip_seasonality)
    clim_cols = ['era5_mean_2m_air_temp_k','era5_total_precipitation_mm',
                 'temp_seasonality','precip_seasonality']
    for c in clim_cols:
        col = pd.to_numeric(sample_df[c], errors='coerce').values
        col = np.where(np.isfinite(col), col, np.nanmedian(col[np.isfinite(col)]) if np.any(np.isfinite(col)) else 0.0)
        Z = np.hstack([Z, col.reshape(-1,1)])

    if level == 4:
        return Z

    # L5: + Shannon diversity + top-8 phylum RAs
    shannon_col = pd.to_numeric(sample_df['shannon'], errors='coerce').values
    shannon_col = np.where(np.isfinite(shannon_col), shannon_col,
                           np.nanmedian(shannon_col[np.isfinite(shannon_col)]) if np.any(np.isfinite(shannon_col)) else 0.0)
    Z = np.hstack([Z, shannon_col.reshape(-1,1)])
    for pc in phyl_cols:
        col = pd.to_numeric(sample_df[pc], errors='coerce').fillna(0.0).values
        Z = np.hstack([Z, col.reshape(-1,1)])

    return Z  # L5


In [6]:
# Run FWL for EUR: all KOs × EUR_METALS × L0-L5
print('Running EUR FWL...')
eur_results = []
for metal in EUR_METALS:
    metal_arr = pd.to_numeric(eur_base[metal], errors='coerce').values
    metal_arr = np.where(metal_arr > 0, metal_arr, np.nan)
    log_metal = np.log10(metal_arr)
    iqr_log = np.nanpercentile(log_metal, 75) - np.nanpercentile(log_metal, 25)

    for level in range(6):  # L0-L5
        Z = build_Z(eur_base, level)
        cwm_vals = cwm_eur.values  # (n_samples, n_KOs)
        valid = np.isfinite(log_metal) & np.all(np.isfinite(Z), axis=1)
        n_valid = int(valid.sum())
        if n_valid < 30:
            print(f'  EUR {metal} L{level}: n={n_valid} < 30, skip')
            continue

        betas, se, t_stat = fwl_all_kos(
            log_metal[valid], cwm_vals[valid], Z[valid])
        dof = max(n_valid - Z.shape[1] - 1, 1)
        pvals = 2 * t_dist.sf(np.abs(t_stat), df=dof)

        for i, ko in enumerate(ko_ids):
            eur_results.append({
                'region': 'EUR', 'metal': metal, 'level': f'L{level}', 'ko_id': ko,
                'n': n_valid, 'beta': betas[i], 'se': se[i],
                't_stat': t_stat[i], 'p': pvals[i],
                'beta_per_iqr': betas[i] * iqr_log,
            })
        print(f'  EUR {metal} L{level}: n={n_valid}')

eur_df = pd.DataFrame(eur_results)
print(f'EUR results: {len(eur_df):,} rows')


Running EUR FWL...
  EUR As L0: n=921


  EUR As L1: n=921


  EUR As L2: n=921


  EUR As L3: n=921


  EUR As L4: n=921


...

  EUR Zn L2: n=921


  EUR Zn L3: n=921


  EUR Zn L4: n=921


  EUR Zn L5: n=921


EUR results: 273,672 rows


In [7]:
# BH-FDR within each EUR metal × level
def apply_fdr(df):
    df = df.copy()
    df['q_bh'] = np.nan
    for (metal, level), grp in df.groupby(['metal','level']):
        valid_p = np.isfinite(grp['p'].values)
        q = np.full(len(grp), np.nan)
        if valid_p.sum() > 0:
            _, q[valid_p], _, _ = multipletests(grp['p'].values[valid_p], method='fdr_bh')
        df.loc[grp.index, 'q_bh'] = q
    return df

eur_df = apply_fdr(eur_df)
eur_L1 = eur_df[eur_df['level']=='L1']
eur_hits = eur_L1[eur_L1['q_bh'] < 0.05]
print(f'EUR L1 FDR hits: {len(eur_hits)} across {eur_hits["metal"].nunique()} metals, {eur_hits["ko_id"].nunique()} KOs')
print(eur_hits['metal'].value_counts().to_string())

eur_df.attrs = {}
eur_df.to_parquet(DATA / 'nb04_eur_fwl.parquet', index=False)
print('Saved nb04_eur_fwl.parquet')


EUR L1 FDR hits: 311 across 7 metals, 145 KOs
metal
Zn    98
Pb    56
Cr    46
As    44
Ni    34
Cd    31
Cu     2
Saved nb04_eur_fwl.parquet


In [8]:
# Run FWL for AUS: all KOs × AUS_METALS × L0-L5
print('Running AUS FWL...')
# Align AUS CWM to aus_base index (may differ from eur ko_ids if KOs differ)
# Use intersection of KOs
aus_ko_ids = cwm_aus.columns.tolist()
shared_kos = [k for k in ko_ids if k in cwm_aus.columns]
cwm_aus_aligned = cwm_aus[shared_kos].reindex(aus_base.index).fillna(0.0)
print(f'AUS CWM shape (shared KOs): {cwm_aus_aligned.shape}')

aus_results = []
for metal in AUS_METALS:
    metal_arr = pd.to_numeric(aus_base[metal], errors='coerce').values
    metal_arr = np.where(metal_arr > 0, metal_arr, np.nan)
    log_metal = np.log10(metal_arr)
    iqr_log = np.nanpercentile(log_metal, 75) - np.nanpercentile(log_metal, 25)

    for level in range(6):
        Z = build_Z(aus_base, level)
        cwm_vals = cwm_aus_aligned.values
        valid = np.isfinite(log_metal) & np.all(np.isfinite(Z), axis=1)
        n_valid = int(valid.sum())
        if n_valid < 30:
            print(f'  AUS {metal} L{level}: n={n_valid} < 30, skip')
            continue

        betas, se, t_stat = fwl_all_kos(
            log_metal[valid], cwm_vals[valid], Z[valid])
        dof = max(n_valid - Z.shape[1] - 1, 1)
        pvals = 2 * t_dist.sf(np.abs(t_stat), df=dof)

        for i, ko in enumerate(shared_kos):
            aus_results.append({
                'region': 'AUS', 'metal': metal, 'level': f'L{level}', 'ko_id': ko,
                'n': n_valid, 'beta': betas[i], 'se': se[i],
                't_stat': t_stat[i], 'p': pvals[i],
                'beta_per_iqr': betas[i] * iqr_log,
            })
        print(f'  AUS {metal} L{level}: n={n_valid}')

aus_df = pd.DataFrame(aus_results)
aus_df = apply_fdr(aus_df)
aus_L1 = aus_df[aus_df['level']=='L1']
aus_hits = aus_L1[aus_L1['q_bh'] < 0.05]
print(f'\nAUS L1 FDR hits: {len(aus_hits)} across {aus_hits["metal"].nunique()} metals, {aus_hits["ko_id"].nunique()} KOs')
print(aus_hits['metal'].value_counts().to_string())

aus_df.attrs = {}
aus_df.to_parquet(DATA / 'nb04_aus_fwl.parquet', index=False)
print('Saved nb04_aus_fwl.parquet')


Running AUS FWL...
AUS CWM shape (shared KOs): (236, 6431)


  AUS As L0: n=221
  AUS As L1: n=221


  AUS As L2: n=221
  AUS As L3: n=221
  AUS As L4: n=221
  AUS As L5: n=221
  AUS Cr L0: n=236


  AUS Cr L1: n=236
  AUS Cr L2: n=236


  AUS Cr L3: n=236


...

  AUS Ni L5: n=234
  AUS Pb L0: n=236
  AUS Pb L1: n=236


  AUS Pb L2: n=236
  AUS Pb L3: n=236
  AUS Pb L4: n=236
  AUS Pb L5: n=236
  AUS Zn L0: n=232
  AUS Zn L1: n=232


  AUS Zn L2: n=232
  AUS Zn L3: n=232


  AUS Zn L4: n=232
  AUS Zn L5: n=232



AUS L1 FDR hits: 0 across 0 metals, 0 KOs
Series([], )
Saved nb04_aus_fwl.parquet


In [9]:
# Replication overlap: USA L1 hits that also appear in EUR or AUS at L1 (same direction)
def overlap_stats(region_hits, region_name, metals_tested):
    results = []
    for metal in metals_tested:
        usa_m = usa_L1_hits[usa_L1_hits['metal']==metal]
        reg_m = region_hits[region_hits['metal']==metal]
        if len(usa_m) == 0 or len(reg_m) == 0:
            continue
        reg_hit_kos = set(reg_m['ko_id'])
        usa_hit_kos = set(usa_m['ko_id'])
        overlap = usa_hit_kos & reg_hit_kos
        # Direction concordance among overlapping hits
        conc = 0
        for ko in overlap:
            usa_dir = int(usa_m[usa_m['ko_id']==ko]['usa_direction'].iloc[0])
            reg_dir = int(np.sign(reg_m[reg_m['ko_id']==ko]['beta_per_iqr'].iloc[0]))
            if usa_dir == reg_dir:
                conc += 1
        results.append({
            'region': region_name, 'metal': metal,
            'n_usa_hits': len(usa_hit_kos),
            'n_reg_hits': len(reg_hit_kos),
            'n_overlap': len(overlap),
            'n_concordant': conc,
            'pct_overlap': 100*len(overlap)/len(usa_hit_kos) if usa_hit_kos else 0,
            'pct_concordant': 100*conc/len(overlap) if overlap else np.nan,
        })
    return pd.DataFrame(results)

eur_overlap = overlap_stats(eur_hits, 'EUR', EUR_METALS)
aus_overlap = overlap_stats(aus_hits, 'AUS', AUS_METALS)
combined_overlap = pd.concat([eur_overlap, aus_overlap], ignore_index=True)

print('=== Replication overlap (USA L1 hits → EUR/AUS L1) ===')
print(combined_overlap.to_string(index=False))


=== Replication overlap (USA L1 hits → EUR/AUS L1) ===
region metal  n_usa_hits  n_reg_hits  n_overlap  n_concordant  pct_overlap  pct_concordant
   EUR    As          41          44         38            38    92.682927           100.0
   EUR    Cd          10          31         10            10   100.000000           100.0
   EUR    Cr          38          46         36            36    94.736842           100.0
   EUR    Ni          38          34         32            32    84.210526           100.0
   EUR    Pb          14          56         14            14   100.000000           100.0
   EUR    Zn          76          98         56            56    73.684211           100.0


In [10]:
# Hit counts across all levels for EUR and AUS
all_df = pd.concat([eur_df, aus_df], ignore_index=True)
all_hits_by_level = (all_df[all_df['q_bh'] < 0.05]
                     .groupby(['region','metal','level']).size()
                     .reset_index(name='n_hits'))

print('=== Hit counts by region × metal × level ===')
print(all_hits_by_level.to_string(index=False))


=== Hit counts by region × metal × level ===
region metal level  n_hits
   EUR    As    L0     484
   EUR    As    L1      44
   EUR    As    L2      53
   EUR    As    L3      65
   EUR    As    L4      36
   EUR    As    L5      36
   EUR    Cd    L0      39
   EUR    Cd    L1      31
   EUR    Cd    L2      12
   EUR    Cd    L3      22
   EUR    Cd    L4      60
   EUR    Cd    L5      15
   EUR    Cr    L0      46
   EUR    Cr    L1      46
   EUR    Cr    L2      41
   EUR    Cr    L3      42
   EUR    Cr    L4      51
   EUR    Cr    L5      47
   EUR    Cu    L0       3
   EUR    Cu    L1       2
   EUR    Cu    L2       6
   EUR    Cu    L4       1
   EUR    Ni    L0      78
   EUR    Ni    L1      34
   EUR    Ni    L4     113
   EUR    Ni    L5      39
   EUR    Pb    L0      68
   EUR    Pb    L1      56
   EUR    Pb    L2      57
   EUR    Pb    L3      14
   EUR    Pb    L4      28
   EUR    Pb    L5      28
   EUR    Zn    L0     104
   EUR    Zn    L1      98
   EUR    

In [11]:
# pH positive control: FWL with pH as exposure (EUR and AUS)
# pH is the strongest known soil microbiome predictor;
# finding many FDR hits here validates the FWL engine recovers real signal.
print('=== pH Positive Control ===')

def build_Z_noph(df, level):
    """Build Z without pH -- for when pH is the exposure."""
    n = len(df)
    Z = np.ones((n, 1))
    if level == 0:
        return Z
    for c in ['clay_pct','som_pct','bulk_density','lith_dist_deg']:
        col = pd.to_numeric(df[c], errors='coerce').values
        col = np.where(np.isfinite(col), col,
                       np.nanmedian(col[np.isfinite(col)]) if np.any(np.isfinite(col)) else 0.0)
        Z = np.hstack([Z, col.reshape(-1, 1)])
    return Z

ph_ctrl_rows = []
for region_name, region_base, region_cwm in [
        ('EUR', eur_base, cwm_eur), ('AUS', aus_base, cwm_aus)]:
    X_ph = pd.to_numeric(region_base['ph_best'], errors='coerce').values
    for z_level, label in [(0, 'intercept only'), (1, '+soil/lith')]:
        Z = build_Z_noph(region_base, z_level)
        valid = np.isfinite(X_ph) & np.all(np.isfinite(Z), axis=1)
        n_valid = int(valid.sum())
        if n_valid < 30:
            print(f'{region_name} {label}: n={n_valid} < 30, skip')
            continue
        betas, se, t_stat = fwl_all_kos(X_ph[valid], region_cwm.values[valid], Z[valid])
        dof = max(n_valid - Z.shape[1] - 1, 1)
        pvals = 2 * t_dist.sf(np.abs(t_stat), df=dof)
        finite_p = np.isfinite(pvals)
        _, q_bh, _, _ = multipletests(pvals[finite_p], method='fdr_bh')
        n_hits = int((q_bh < 0.05).sum())
        print(f'{region_name} pH ({label}): n={n_valid}, FDR hits = {n_hits}/{len(ko_ids)}')
        ph_ctrl_rows.append({'region': region_name, 'z': label, 'n': n_valid, 'n_hits': n_hits})

print(pd.DataFrame(ph_ctrl_rows).to_string(index=False))


=== pH Positive Control ===
EUR pH (intercept only): n=921, FDR hits = 3175/6516


EUR pH (+soil/lith): n=921, FDR hits = 863/6516
AUS pH (intercept only): n=236, FDR hits = 2918/6516
AUS pH (+soil/lith): n=236, FDR hits = 44/6516
region              z   n  n_hits
   EUR intercept only 921    3175
   EUR     +soil/lith 921     863
   AUS intercept only 236    2918
   AUS     +soil/lith 236      44


In [12]:
# Reverse direction: CWM -> metal, RidgeCV + spatial block CV (5 KMeans clusters)
# Mirrors NB02 reverse analysis; tests EUR/AUS out-of-fold predictability.
from sklearn.linear_model import RidgeCV
from sklearn.cluster import KMeans
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score

def spatial_block_r2(cwm_mat, y, coords, n_clusters=5):
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    groups = km.fit_predict(coords)
    clf = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
    logo = LeaveOneGroupOut()
    try:
        scores = cross_val_score(
            clf, cwm_mat, y,
            cv=logo.split(cwm_mat, y, groups),
            scoring='r2', n_jobs=1)
        return float(np.mean(scores))
    except Exception as e:
        print(f'  error: {e}')
        return np.nan

print('=== Reverse: CWM -> metal (RidgeCV + spatial block CV) ===')
reverse_rows = []

for region_name, region_base, region_cwm, metals in [
        ('EUR', eur_base, cwm_eur, EUR_METALS),
        ('AUS', aus_base, cwm_aus, AUS_METALS)]:
    coords = region_base[['lat', 'lon']].values
    for metal in metals:
        if metal not in region_base.columns:
            print(f'{region_name} {metal}: column not found, skip')
            continue
        y = pd.to_numeric(region_base[metal], errors='coerce').values
        valid = np.isfinite(y) & np.all(np.isfinite(region_cwm.values), axis=1)
        n_valid = int(valid.sum())
        if n_valid < 50:
            print(f'{region_name} {metal}: n={n_valid} < 50, skip')
            continue
        y_v = np.log10(y[valid].clip(min=1e-6))   # log-transform same as NB02
        cwm_v = region_cwm.values[valid]
        coords_v = coords[valid]
        r2 = spatial_block_r2(cwm_v, y_v, coords_v)
        print(f'{region_name} {metal}: n={n_valid}, spatial R2={r2:.3f}')
        reverse_rows.append({'region': region_name, 'metal': metal, 'n': n_valid, 'r2_spatial': r2})

print('\nSummary:')
print(pd.DataFrame(reverse_rows).to_string(index=False))


=== Reverse: CWM -> metal (RidgeCV + spatial block CV) ===


EUR As: n=921, spatial R2=-0.434


EUR Cd: n=921, spatial R2=-0.494


EUR Cr: n=921, spatial R2=-0.165


EUR Cu: n=921, spatial R2=-0.183


...

AUS Cr: n=236, spatial R2=-0.427


AUS Cu: n=232, spatial R2=-0.255


AUS Ni: n=234, spatial R2=-0.262


AUS Pb: n=236, spatial R2=-0.268


AUS Zn: n=232, spatial R2=-0.832

Summary:
region metal   n  r2_spatial
   EUR    As 921   -0.434471
   EUR    Cd 921   -0.494150
   EUR    Cr 921   -0.164838
   EUR    Cu 921   -0.182796
   EUR    Ni 921   -0.234614
   EUR    Pb 921   -0.790540
   EUR    Zn 921   -0.242500
   AUS    As 221   -0.402351
   AUS    Cr 236   -0.427054
   AUS    Cu 232   -0.255460
   AUS    Ni 234   -0.262433
   AUS    Pb 236   -0.268225
   AUS    Zn 232   -0.832481


In [13]:
# Figure: replication overlap heatmap (EUR and AUS side by side)
fig, axes = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H))

for ax, (reg_overlap, reg_name, metals) in zip(
        axes, [(eur_overlap,'EUR',EUR_METALS),(aus_overlap,'AUS',AUS_METALS)]):
    if len(reg_overlap) == 0:
        ax.text(0.5, 0.5, 'No overlap', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(reg_name, fontsize=10)
        continue

    pct = reg_overlap.set_index('metal')['pct_overlap'].reindex(metals).fillna(0)
    conc = reg_overlap.set_index('metal')['pct_concordant'].reindex(metals)
    n_ov = reg_overlap.set_index('metal')['n_overlap'].reindex(metals).fillna(0).astype(int)

    x = np.arange(len(metals))
    bars = ax.bar(x, pct.values, color=PALETTE[0], edgecolor='k', linewidth=0.5)
    for xi, (pv, cv, nv) in enumerate(zip(pct.values, conc.values, n_ov.values)):
        if nv > 0:
            label = f'n={nv}\n{cv:.0f}%conc' if not np.isnan(cv) else f'n={nv}'
            ax.text(xi, pv + 0.5, label, ha='center', va='bottom', fontsize=6, color='#808080')

    ax.set_xticks(x)
    ax.set_xticklabels(metals, fontsize=8)
    ax.set_xlabel('Metal')
    ax.set_ylabel('% USA L1 hits replicated' if reg_name=='EUR' else '')
    ax.set_title(f'{reg_name} replication of USA L1 hits', fontsize=10)
    ax.set_ylim(0, max(pct.max() * 1.3, 5))

fig.suptitle('EUR and AUS replication of NB02 USA L1 FDR hits', y=1.02)
fig.tight_layout()
save(fig, FIGS / 'fig_nb04_replication_overlap')


**Figure:** Percentage of NB02 USA L1 FDR hits that replicate in EUR (GEMAS) and AUS (NGSA) at L1. Numbers show count of replicated hits and direction concordance (% same sign as USA β).


In [14]:
# Figure: hit count heatmap across metals × levels for EUR and AUS
level_order = ['L0','L1','L2','L3','L4','L5']

fig, axes = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H * 1.1))
import matplotlib.colors as mcolors

for ax, (reg_df, reg_name) in zip(axes, [(eur_df,'EUR'), (aus_df,'AUS')]):
    reg_hits = reg_df[reg_df['q_bh'] < 0.05]
    pivot = (reg_hits.groupby(['metal','level']).size()
             .unstack(fill_value=0)
             .reindex(columns=level_order, fill_value=0))
    if pivot.empty:
        ax.text(0.5, 0.5, 'No hits', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(reg_name, fontsize=10)
        continue
    pivot['total'] = pivot.sum(axis=1)
    pivot = pivot.sort_values('total', ascending=False).drop(columns='total')

    mat = pivot.values.astype(float)
    mat_disp = np.where(mat == 0, np.nan, mat)
    if mat.max() > 1:
        im = ax.imshow(mat_disp, aspect='auto', cmap='YlOrRd',
                       norm=mcolors.LogNorm(vmin=1, vmax=max(mat.max(), 2)))
        plt.colorbar(im, ax=ax, label='FDR hits', shrink=0.8)
    else:
        ax.imshow(mat_disp, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)

    ax.set_xticks(range(len(level_order)))
    ax.set_xticklabels(level_order, fontsize=8)
    ax.set_yticks(range(len(pivot)))
    ax.set_yticklabels(pivot.index, fontsize=7)
    ax.set_xlabel('Causal level')
    ax.set_ylabel('Metal')
    ax.set_title(f'{reg_name} (n={int(reg_df["n"].max())})', fontsize=10)

    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            if v > 0:
                ax.text(j, i, int(v), ha='center', va='center', fontsize=6,
                        color='black' if v < 50 else 'white')

fig.suptitle('FDR hit counts per metal × causal level', y=1.02)
fig.tight_layout()
save(fig, FIGS / 'fig_nb04_hits_heatmap')


**Figure:** FDR hit counts in EUR and AUS across metals × causal levels. Compare with NB02 USA pattern to assess cross-regional consistency.


In [15]:
print('=== NB04 Summary ===')
print(f'EUR: {len(eur_base)} samples, {len(EUR_METALS)} metals tested')
print(f'  L1 FDR hits: {len(eur_hits)}')
print(f'  Hit metals: {eur_hits["metal"].value_counts().to_dict()}')
print()
print(f'AUS: {len(aus_base)} samples, {len(AUS_METALS)} metals tested')
print(f'  L1 FDR hits: {len(aus_hits)}')
print(f'  Hit metals: {aus_hits["metal"].value_counts().to_dict()}')
print()
print('Replication of USA L1 hits:')
print(combined_overlap[['region','metal','n_usa_hits','n_overlap','pct_overlap','pct_concordant']].to_string(index=False))


=== NB04 Summary ===
EUR: 921 samples, 7 metals tested
  L1 FDR hits: 311
  Hit metals: {'Zn': 98, 'Pb': 56, 'Cr': 46, 'As': 44, 'Ni': 34, 'Cd': 31, 'Cu': 2}

AUS: 236 samples, 6 metals tested
  L1 FDR hits: 0
  Hit metals: {}

Replication of USA L1 hits:
region metal  n_usa_hits  n_overlap  pct_overlap  pct_concordant
   EUR    As          41         38    92.682927           100.0
   EUR    Cd          10         10   100.000000           100.0
   EUR    Cr          38         36    94.736842           100.0
   EUR    Ni          38         32    84.210526           100.0
   EUR    Pb          14         14   100.000000           100.0
   EUR    Zn          76         56    73.684211           100.0
